In [1]:
import requests

url = "https://www.gov.br/trabalho-e-emprego/pt-br/assuntos/inspecao-do-trabalho/areas-de-atuacao/cadastro_de_empregadores.pdf"

response = requests.get(url)

# Save file
with open("bronze/cadastro_empregadores.pdf", "wb") as f:
    f.write(response.content)

In [2]:
import pdfplumber
import pandas as pd

tables = []

with pdfplumber.open("bronze/cadastro_empregadores.pdf") as pdf:
    for page in pdf.pages:
        table = page.extract_table()
        if table:
            # Find the header row (starts with 'ID')
            header_idx = next((i for i, row in enumerate(table) if row[0] and row[0].strip() == "ID"), None)
            if header_idx is not None:
                header = table[header_idx]
                data_rows = table[header_idx + 1:]
                df = pd.DataFrame(data_rows, columns=header)
                tables.append(df)

# Combine into one DataFrame
full_table = pd.concat(tables, ignore_index=True)

In [4]:
full_table.to_csv("silver/cadastro_empregadores.csv", index=False, encoding="utf-8")